In [31]:
# Adjusted version of the valuation model with rounding only at the final stage

def compute_model_no_intermediate_rounding():
    # Inputs from the provided Excel table
    wacc = 0.105
    tax_rate = 0.25
    opm = 0.4
    ppe_to_rev_ratio = 0.3
    dep_rate = 0.12
    next_stage_ronic = 0.13
    terminal_roic = 0.13

    
    non_op_inc_growth = 0.05
    non_op_inc_terminal_growth = 0.03
    terminal_growth = 0.05
    next_stage_growth = 0.10
    next_stage_period = 10
    
    ttm_revenue = 351.56
    non_op_inc_ttm = 13.575
    debt = 269.02
    current_market_cap = 2284.12
    current_market_price = 429.45
    
    growth_rate_stage1 = 0.11


    # Initialize arrays
    revenues = [ttm_revenue]
    ebitda = []
    net_ppe = []
    depreciation = []
    noplat = []
    capex = []
    fcff = []
    non_op_income = [non_op_inc_ttm]

    # Compute for Years 1 to 11
    for year in range(1, 12):  # Include Year 11 for terminal value calculations
        # Revenue growth
        growth_rate = growth_rate_stage1 if year <= 10 else next_stage_growth
        revenues.append(revenues[-1] * (1 + growth_rate))
        
        # EBITDA, Net PP&E, Depreciation, NOPLAT, Capex, and FCFF
        if year > 0:
            ebitda.append(revenues[-1] * opm)
            net_ppe.append(revenues[-1] * ppe_to_rev_ratio)
            depreciation.append(net_ppe[-1] * dep_rate)
            noplat.append((ebitda[-1] - depreciation[-1]) * (1 - tax_rate))
            # Capex correction
            if year > 1:
                capex.append(net_ppe[-1] - net_ppe[-2] + depreciation[-1])
            elif year == 1:
                capex.append(net_ppe[-1] - (ttm_revenue * ppe_to_rev_ratio) + depreciation[-1])
            fcff.append(noplat[-1] + depreciation[-1] - capex[-1])

        # Non-Operating Income
        if year <= 10:
            non_op_income.append(non_op_income[-1] * (1 + non_op_inc_growth))
        else:
            non_op_income.append(non_op_income[-1] * (1 + non_op_inc_terminal_growth))

    # Discount factors for Years 1 to 10
    discount_factors = [(1 / (1 + wacc) ** year) for year in range(1, 12)]

    # PV(FCFF) for Years 1 to 10
    pv_fcff = [fcf * df for fcf, df in zip(fcff[:10], discount_factors[:10])]
    pv_fcff_sum = sum(pv_fcff)

    # Continue Value for Operating Business using the 2-stage model
    noplat_11 = noplat[-1]  # Year 11 NOPLAT
    print('noplat_11-',noplat_11)
    first_stage_value = (
        noplat_11
        * (1 - next_stage_growth / next_stage_ronic) / (wacc - next_stage_growth)
        * (1 - ((1 + next_stage_growth) / (1 + wacc)) ** next_stage_period)
    )
    second_stage_value = (
        noplat_11
        * ((1 + next_stage_growth) ** next_stage_period)
        * (1 - terminal_growth / terminal_roic) / (wacc - terminal_growth)
        / ((1 + wacc) ** next_stage_period)
    )
    continue_value_operating = first_stage_value + second_stage_value
    pv_cont_value_operating = continue_value_operating * discount_factors[9]

    # Continue Value for Non-Operating Business (Year 10 Non-Op Income)
    non_op_inc_10 = non_op_income[10]  # Year 10 Non-Op Income
    continue_value_non_op = non_op_inc_10 * (1 + non_op_inc_terminal_growth) / (wacc - non_op_inc_terminal_growth)
    pv_cont_value_non_op = continue_value_non_op * discount_factors[9]

    # PV of Non-Operating Income (Years 1 to 10 + Terminal Value)
    pv_non_op_income_10yrs = sum([noi * df for noi, df in zip(non_op_income[:10], discount_factors[:10])])
    pv_non_op_income = pv_non_op_income_10yrs + pv_cont_value_non_op

    # Total Value and Expected Price
    exp_market_cap = pv_fcff_sum + pv_cont_value_operating + pv_non_op_income - debt
    exp_price = exp_market_cap / (current_market_cap / current_market_price)

    # Results with final rounding for display purposes
    return {
        "Intermediate Table": pd.DataFrame({
            "Revenues": revenues[1:11],
            "EBITDA": ebitda[:10],
            "Net PP&E": net_ppe[:10],
            "Depreciation": depreciation[:10],
            "NOPLAT": noplat[:10],
            "Capex": capex[:10],
            "FCFF": fcff[:10],
            "Non-Op Income": non_op_income[1:11],
            "Discount Factors": discount_factors[:10],
        }),
        "Cont. Value (Operating)": round(continue_value_operating, 0),
        "Cont. Value (Non-Op)": round(continue_value_non_op, 0),
        "PV(FCFF)": round(pv_fcff_sum, 0),
        "PV(Cont. Value - Operating)": round(pv_cont_value_operating, 0),
        "PV(Cont. Value - Non-Op)": round(pv_cont_value_non_op, 0),
        "PV(FCFF + CV)": round(pv_fcff_sum + pv_cont_value_operating, 0),
        "PV(Non-Op. Inc)": round(pv_non_op_income, 0),
        "Out. Debt": debt,
        "Exp. Market Cap": round(exp_market_cap, 0),
        "Exp. Price": round(exp_price, 0),
    }


# Compute and display results
model_results = compute_model_no_intermediate_rounding()

# Display intermediate valuation table
print("\nIntermediate Valuation Table:")
print(model_results["Intermediate Table"])

# Display final results
print("\nFinal Results:")
print(f"Cont. Value of operating business on 10th year: {model_results['Cont. Value (Operating)']}")
print(f"Cont. Value of non operating business on 10th year: {model_results['Cont. Value (Non-Op)']}")
print(f"PV(FCFF): {model_results['PV(FCFF)']}")
print(f"PV(Cont. Value of operating business): {model_results['PV(Cont. Value - Operating)']}")
print(f"PV(FCFF + CV): {model_results['PV(FCFF + CV)']}")
print(f"PV(Non-Op. Inc): {model_results['PV(Non-Op. Inc)']}")
print(f"Out. Debt: {model_results['Out. Debt']}")
print(f"Exp. Market Cap: {model_results['Exp. Market Cap']}")
print(f"Exp. Price: {model_results['Exp. Price']}")

noplat_11- 299.76752061128605

Intermediate Valuation Table:
     Revenues      EBITDA    Net PP&E  Depreciation      NOPLAT      Capex  \
0  390.231600  156.092640  117.069480     14.048338  106.533227  25.649818   
1  433.157076  173.262830  129.947123     15.593655  118.251882  28.471298   
2  480.804354  192.321742  144.241306     17.308957  131.259589  31.603140   
3  533.692833  213.477133  160.107850     19.212942  145.698144  35.079486   
4  592.399045  236.959618  177.719714     21.326366  161.724939  38.938229   
5  657.562940  263.025176  197.268882     23.672266  179.514683  43.221434   
6  729.894863  291.957945  218.968459     26.276215  199.261298  47.975792   
7  810.183298  324.073319  243.054989     29.166599  221.180040  53.253129   
8  899.303461  359.721384  269.791038     32.374925  245.509845  59.110973   
9  998.226842  399.290737  299.468053     35.936166  272.515928  65.613181   

         FCFF  Non-Op Income  Discount Factors  
0   94.931747      14.253750   